# QUESTÕES DE IMPLEMENTAÇÃO DE CÓDIGO AC01

In [1]:
#### 5 ######
import random

# =====================================================================
# 1. ESTRUTURA DA LISTA ENCADEADA SIMPLES (PARA AS LIGAÇÕES / ARESTAS)
# =====================================================================

class NodoAresta:
    """Nó da Lista Encadeada Simples de Arestas"""
    def __init__(self, id_aresta, destino, custo=1.0, caracteristica=""):
        self.id_aresta = id_aresta
        self.destino = destino
        self.custo = custo
        self.caracteristica = caracteristica
        self.prox = None  # Ponteiro para o próximo nó da lista encadeada simples

class ListaEncadeadaArestas:
    """Lista Encadeada Simples de adjacências de um vértice"""
    def __init__(self):
        self.head = None

    def inserir(self, id_aresta, destino, custo, caracteristica):
        novo_no = NodoAresta(id_aresta, destino, custo, caracteristica)
        novo_no.prox = self.head
        self.head = novo_no

    def buscar(self, id_aresta):
        atual = self.head
        while atual:
            if atual.id_aresta == id_aresta:
                return atual
            atual = atual.prox
        return None

    def alterar(self, id_aresta, novo_custo=None, nova_caracteristica=None):
        no = self.buscar(id_aresta)
        if no:
            if novo_custo is not None:
                no.custo = novo_custo
            if nova_caracteristica is not None:
                no.caracteristica = nova_caracteristica
            return True
        return False

    def remover(self, id_aresta):
        atual = self.head
        anterior = None
        while atual:
            if atual.id_aresta == id_aresta:
                if anterior is None:
                    self.head = atual.prox
                else:
                    anterior.prox = atual.prox
                return True
            anterior = atual
            atual = atual.prox
        return False

    def remover_por_destino(self, destino_id):
        """Remove todas as ligações que apontam para um vértice excluído"""
        atual = self.head
        anterior = None
        while atual:
            if atual.destino == destino_id:
                if anterior is None:
                    self.head = atual.prox
                    atual = self.head
                else:
                    anterior.prox = atual.prox
                    atual = anterior.prox
            else:
                anterior = atual
                atual = atual.prox

    def para_lista(self):
        arestas = []
        atual = self.head
        while atual:
            arestas.append(atual)
            atual = atual.prox
        return arestas


# =====================================================================
# 2. ESTRUTURA DO VÉRTICE E DO MULTIGRAFO
# =====================================================================

class Vertice:
    """Elemento do Vetor de Vértices"""
    def __init__(self, id_vertice, rotulo="", custo=0.0):
        self.id_vertice = id_vertice
        self.rotulo = rotulo
        self.custo = custo
        self.lista_adj = ListaEncadeadaArestas()  # Aponta para sua Lista Encadeada

class Multigrafo:
    """Classe Multigrafo baseada em Vetor de Listas Encadeadas Simples"""
    def __init__(self, nome="Multigrafo"):
        self.nome = nome
        self.vertices = []  # Vetor de objetos Vertice
        self.proximo_id_aresta = 1

    def destruir(self):
        """Destrói a estrutura e esvazia o grafo"""
        self.vertices.clear()
        self.proximo_id_aresta = 1

    # --- OPERAÇÕES DE VÉRTICES ---

    def incluir_vertice(self, id_vertice, rotulo="", custo=0.0):
        if self.buscar_vertice(id_vertice) is not None:
            return False
        novo_v = Vertice(id_vertice, rotulo, custo)
        self.vertices.append(novo_v)
        return True

    def buscar_vertice(self, id_vertice):
        for v in self.vertices:
            if v.id_vertice == id_vertice:
                return v
        return None

    def alterar_vertice(self, id_vertice, novo_rotulo=None, novo_custo=None):
        v = self.buscar_vertice(id_vertice)
        if v:
            if novo_rotulo is not None:
                v.rotulo = novo_rotulo
            if novo_custo is not None:
                v.custo = novo_custo
            return True
        return False

    def remover_vertice(self, id_vertice):
        v = self.buscar_vertice(id_vertice)
        if not v:
            return False

        # Remove o vértice do vetor
        self.vertices.remove(v)

        # Remove todas as arestas de outros vértices que apontavam para o vértice excluído
        for outro_v in self.vertices:
            outro_v.lista_adj.remover_por_destino(id_vertice)
        return True

    # --- OPERAÇÕES DE LIGAÇÕES / ARESTAS (SUPORTA MULTIPLAS LIGAÇÕES) ---

    def incluir_aresta(self, origem_id, destino_id, custo=1.0, caracteristica="", id_aresta=None):
        v_origem = self.buscar_vertice(origem_id)
        v_destino = self.buscar_vertice(destino_id)

        if not v_origem or not v_destino:
            return None

        if id_aresta is None:
            id_aresta = self.proximo_id_aresta
            self.proximo_id_aresta += 1

        # Inserção na lista encadeada simples da origem (admite múltiplas arestas entre u e v)
        v_origem.lista_adj.inserir(id_aresta, destino_id, custo, caracteristica)
        return id_aresta

    def buscar_aresta(self, id_aresta):
        for v in self.vertices:
            aresta = v.lista_adj.buscar(id_aresta)
            if aresta:
                return aresta, v.id_vertice
        return None, None

    def alterar_aresta(self, id_aresta, novo_custo=None, nova_caracteristica=None):
        aresta, origem_id = self.buscar_aresta(id_aresta)
        if aresta:
            v_origem = self.buscar_vertice(origem_id)
            return v_origem.lista_adj.alterar(id_aresta, novo_custo, nova_caracteristica)
        return False

    def remover_aresta(self, id_aresta):
        aresta, origem_id = self.buscar_aresta(id_aresta)
        if aresta:
            v_origem = self.buscar_vertice(origem_id)
            return v_origem.lista_adj.remover(id_aresta)
        return False

    # --- MOSTRAR E PREENCHIMENTO ---

    def mostrar(self):
        print(f"\n=================== {self.nome} ===================")
        if not self.vertices:
            print("Grafo vazio.")
            print("=====================================================")
            return

        print(f"Total de Vértices: {len(self.vertices)}")
        for v in self.vertices:
            print(f"\n-> [Vértice ID: {v.id_vertice}] | Rótulo: '{v.rotulo}' | Custo: {v.custo}")
            arestas = v.lista_adj.para_lista()
            if not arestas:
                print("   └── (Sem ligações de saída)")
            else:
                for a in arestas:
                    print(f"   └── (Aresta ID: {a.id_aresta}) ---> Destino: {a.destino} | Custo: {a.custo} | Caract: '{a.caracteristica}'")
        print("=====================================================")

    def preenchimento_automatico(self, num_vertices=4, num_arestas=6):
        """Gera um grafo automaticamente com atributos aleatórios"""
        self.destruir()

        tipos_v = ["Servidor", "Roteador", "Switch", "DataCenter"]
        for i in range(1, num_vertices + 1):
            self.incluir_vertice(i, f"{random.choice(tipos_v)}_{i}", round(random.uniform(10, 50), 2))

        tipos_a = ["Fibra", "Rádio", "Cabo", "Satélite"]
        for _ in range(num_arestas):
            u = random.randint(1, num_vertices)
            v = random.randint(1, num_vertices)
            self.incluir_aresta(u, v, round(random.uniform(1.0, 20.0), 2), random.choice(tipos_a))

    def obter_estatisticas(self):
        total_v = len(self.vertices)
        total_a = 0
        custo_a = 0.0
        for v in self.vertices:
            arestas = v.lista_adj.para_lista()
            total_a += len(arestas)
            custo_a += sum(a.custo for a in arestas)

        return {
            "Vertices": total_v,
            "Arestas": total_a,
            "Custo Total Arestas": round(custo_a, 2),
            "Grau Médio Saída": round(total_a / total_v, 2) if total_v > 0 else 0
        }


# =====================================================================
# 3. FUNÇÃO DE COMPARAÇÃO ENTRE GRAFOS
# =====================================================================

def comparar_grafos(g1: Multigrafo, g2: Multigrafo):
    e1 = g1.obter_estatisticas()
    e2 = g2.obter_estatisticas()

    print("\n" + "="*55)
    print(f"          COMPARAÇÃO: {g1.nome} vs {g2.nome}")
    print("="*55)
    print(f"{'Métrica':<25} | {g1.nome:<12} | {g2.nome:<12}")
    print("-" * 55)
    for chave in e1:
        print(f"{chave:<25} | {str(e1[chave]):<12} | {str(e2[chave]):<12}")
    print("="*55)


#======== IMPLEMENTAÇÃO ==========#
# 1. Criando Grafo 1 (Manual)
g1 = Multigrafo("Grafo_Rede_Manual")

# Inserção manual de Vértices
g1.incluir_vertice(1, "Roteador_Central", 150.0)
g1.incluir_vertice(2, "Switch_A", 80.0)
g1.incluir_vertice(3, "Servidor_Web", 200.0)

# Inserção manual de Múltiplas Ligações (Multigrafo) entre os mesmos nós
a1 = g1.incluir_aresta(1, 2, custo=10.5, caracteristica="Link Principal Fibra")
a2 = g1.incluir_aresta(1, 2, custo=25.0, caracteristica="Link Redundante Rádio")
a3 = g1.incluir_aresta(2, 3, custo=5.0, caracteristica="Cabo UTP Cat6")

# Alteração de dados
g1.alterar_vertice(2, novo_rotulo="Switch_A_Atualizado", novo_custo=95.0)
g1.alterar_aresta(a1, novo_custo=8.0, nova_caracteristica="Fibra Optica 10Gbps")

# Exibição do Grafo 1
g1.mostrar()

# 2. Criando Grafo 2 (Automático)
g2 = Multigrafo("Grafo_Rede_Auto")
g2.preenchimento_automatico(num_vertices=5, num_arestas=10)

# Exibição do Grafo 2
g2.mostrar()

# 3. Comparação Direta
comparar_grafos(g1, g2)


=================== Grafo_Rede_Manual ===================
Total de Vértices: 3

-> [Vértice ID: 1] | Rótulo: 'Roteador_Central' | Custo: 150.0
   └── (Aresta ID: 2) ---> Destino: 2 | Custo: 25.0 | Caract: 'Link Redundante Rádio'
   └── (Aresta ID: 1) ---> Destino: 2 | Custo: 8.0 | Caract: 'Fibra Optica 10Gbps'

-> [Vértice ID: 2] | Rótulo: 'Switch_A_Atualizado' | Custo: 95.0
   └── (Aresta ID: 3) ---> Destino: 3 | Custo: 5.0 | Caract: 'Cabo UTP Cat6'

-> [Vértice ID: 3] | Rótulo: 'Servidor_Web' | Custo: 200.0
   └── (Sem ligações de saída)

=================== Grafo_Rede_Auto ===================
Total de Vértices: 5

-> [Vértice ID: 1] | Rótulo: 'Servidor_1' | Custo: 13.45
   └── (Aresta ID: 8) ---> Destino: 2 | Custo: 6.42 | Caract: 'Rádio'
   └── (Aresta ID: 6) ---> Destino: 4 | Custo: 10.51 | Caract: 'Cabo'
   └── (Aresta ID: 5) ---> Destino: 4 | Custo: 12.49 | Caract: 'Satélite'

-> [Vértice ID: 2] | Rótulo: 'Switch_2' | Custo: 15.43
   └── (Sem ligações de saída)

-> [Vértice ID:

In [2]:
#==========6===========#


from collections import defaultdict

class GrafoDAG:
    def __init__(self, num_vertices):
        self.V = num_vertices
        # Lista de Adjacência usando dicionário de listas
        self.adj = defaultdict(list)

    def adicionar_aresta(self, u, v):
        self.adj[u].append(v)

    def _dfs_topologica(self, v, visitado, pilha):
        visitado[v] = True

        # Visita recursivamente todos os vértices dependentes
        for vizinho in self.adj[v]:
            if not visitado[vizinho]:
                self._dfs_topologica(vizinho, visitado, pilha)

        # Empilha o vértice apenas após processar todos os seus descendentes (pós-ordem)
        pilha.append(v)

    def ordenacao_topologica(self):
        visitado = [False] * self.V
        pilha = []

        # Executa a busca em profundidade para todos os componentes desconexos
        for i in range(self.V):
            if not visitado[i]:
                self._dfs_topologica(i, visitado, pilha)

        # O resultado topológico é a ordem de desempilhamento (pós-ordem invertida)
        return pilha[::-1]


g = GrafoDAG(15)
arestas = [
    (0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6),
    (3, 7), (4, 7), (4, 8), (5, 8), (5, 9), (6, 9),
    (7, 10), (8, 10), (8, 11), (9, 11), (10, 12), (11, 13),
    (12, 14), (13, 14)
]
for u, v in arestas:
    g.adicionar_aresta(u, v)
rotulacao = g.ordenacao_topologica()
print("Resultado da Rotulação Topológica:", rotulacao)

Resultado da Rotulação Topológica: [0, 2, 6, 5, 9, 1, 4, 8, 11, 13, 3, 7, 10, 12, 14]


In [3]:
#========7=========

# 1. Construção do grafo (DAG) com pelo menos 15 vértices, usando a estrutura da Questão 5
g_topo = Multigrafo("Grafo_Rotulacao_PosOrdem")
for i in range(15):
    g_topo.incluir_vertice(i, f"V{i}")

arestas_dag = [
    (0,1),(0,2),(1,3),(1,4),(2,5),(2,6),
    (3,7),(4,7),(4,8),(5,8),(5,9),(6,9),
    (7,10),(8,10),(8,11),(9,11),(10,12),(11,13),
    (12,14),(13,14)
]
for u, v in arestas_dag:
    g_topo.incluir_aresta(u, v)

# 2. Travessia em pós-ordem: o vértice só é processado DEPOIS de visitar todos os seus descendentes
def dfs_pos_ordem(grafo, inicio_id, visitado, sequencia_pos_ordem):
    visitado.add(inicio_id)
    v = grafo.buscar_vertice(inicio_id)
    for aresta in v.lista_adj.para_lista():
        if aresta.destino not in visitado:
            dfs_pos_ordem(grafo, aresta.destino, visitado, sequencia_pos_ordem)
    sequencia_pos_ordem.append(inicio_id)  # só entra na lista após esgotar os filhos

def rotulacao_topologica_pos_ordem(grafo):
    visitado = set()
    sequencia_pos_ordem = []
    for v in grafo.vertices:
        if v.id_vertice not in visitado:
            dfs_pos_ordem(grafo, v.id_vertice, visitado, sequencia_pos_ordem)
    ordem_topologica = sequencia_pos_ordem[::-1]  # inverter a pós-ordem dá a ordem topológica válida
    rotulos = {vid: pos for pos, vid in enumerate(ordem_topologica)}
    return sequencia_pos_ordem, ordem_topologica, rotulos

# 3. Execução
seq_pos, ordem_topo, rotulos = rotulacao_topologica_pos_ordem(g_topo)

print("=== TRAVESSIA EM PÓS-ORDEM (Questão 7) ===")
print(f"Sequência de finalização (pós-ordem): {seq_pos}\n")
print(f"Ordem topológica obtida (inverso da pós-ordem): {ordem_topo}\n")
print("Rótulos atribuídos aos vértices:")
for v in sorted(rotulos.keys()):
    print(f"  Vértice {v:2d} -> Rótulo Topológico: {rotulos[v]:2d}")

=== TRAVESSIA EM PÓS-ORDEM (Questão 7) ===
Sequência de finalização (pós-ordem): [14, 13, 11, 9, 6, 12, 10, 8, 5, 2, 7, 4, 3, 1, 0]

Ordem topológica obtida (inverso da pós-ordem): [0, 1, 3, 4, 7, 2, 5, 8, 10, 12, 6, 9, 11, 13, 14]

Rótulos atribuídos aos vértices:
  Vértice  0 -> Rótulo Topológico:  0
  Vértice  1 -> Rótulo Topológico:  1
  Vértice  2 -> Rótulo Topológico:  5
  Vértice  3 -> Rótulo Topológico:  2
  Vértice  4 -> Rótulo Topológico:  3
  Vértice  5 -> Rótulo Topológico:  6
  Vértice  6 -> Rótulo Topológico: 10
  Vértice  7 -> Rótulo Topológico:  4
  Vértice  8 -> Rótulo Topológico:  7
  Vértice  9 -> Rótulo Topológico: 11
  Vértice 10 -> Rótulo Topológico:  8
  Vértice 11 -> Rótulo Topológico: 12
  Vértice 12 -> Rótulo Topológico:  9
  Vértice 13 -> Rótulo Topológico: 13
  Vértice 14 -> Rótulo Topológico: 14


In [4]:
#========8=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(10))
ARESTAS = [(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(0,9),(2,5),(4,8)]

# 2. LÓGICA
def construir_adjacencia(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)
    return adj

def subgrafo_maximal_arvore(vertices, arestas):
    """Produz uma árvore geradora de G via DFS: é um subgrafo maximal em forma
    de árvore porque cobre todos os vértices e não dá pra adicionar nenhuma
    aresta de G a ela sem formar ciclo."""
    adj = construir_adjacencia(vertices, arestas)
    visitado = set()
    arestas_arvore = []

    def dfs(u):
        visitado.add(u)
        for v in adj[u]:
            if v not in visitado:
                arestas_arvore.append((u, v))
                dfs(v)

    for v in vertices:
        if v not in visitado:
            dfs(v)  # cobre também grafos desconexos (gera floresta geradora)

    conj_arvore = {tuple(sorted(t)) for t in arestas_arvore}
    arestas_fora = [a for a in arestas if tuple(sorted(a)) not in conj_arvore]
    return arestas_arvore, arestas_fora

# 3. EXECUÇÃO
arvore, fora = subgrafo_maximal_arvore(VERTICES, ARESTAS)

print("=== SUBGRAFO MAXIMAL ÁRVORE (ÁRVORE GERADORA) ===")
print(f"Vértices: {len(VERTICES)} | Arestas originais: {len(ARESTAS)}")
print(f"Arestas da árvore geradora ({len(arvore)}): {arvore}")
print(f"Arestas de G fora da árvore ({len(fora)}): {fora}")
print(f"\nVerificação: |arestas da árvore| = {len(arvore)} (esperado |V|-1 = {len(VERTICES)-1} p/ grafo conexo)")

=== SUBGRAFO MAXIMAL ÁRVORE (ÁRVORE GERADORA) ===
Vértices: 10 | Arestas originais: 13
Arestas da árvore geradora (9): [(0, 1), (1, 3), (3, 2), (2, 5), (5, 4), (4, 8), (8, 7), (7, 6), (8, 9)]
Arestas de G fora da árvore (4): [(0, 2), (3, 4), (5, 6), (0, 9)]

Verificação: |arestas da árvore| = 9 (esperado |V|-1 = 9 p/ grafo conexo)


In [5]:
#========9=========
from collections import defaultdict

# 1. ÁREA DE INPUT EDITÁVEL
ARESTAS_G1 = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(9,0)]  # C10
ARESTAS_G2 = [(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),(16,17),(17,18),(18,19),(19,10)]  # outro C10

# 2. LÓGICA
def construir_adj_set(arestas):
    adj = defaultdict(set)
    vertices = set()
    for u, v in arestas:
        adj[u].add(v); adj[v].add(u)
        vertices.update([u, v])
    return adj, vertices

def grau_sequencia(adj, vertices):
    return sorted(len(adj[v]) for v in vertices)

def sao_isomorfos(arestas_g1, arestas_g2):
    adj1, v1 = construir_adj_set(arestas_g1)
    adj2, v2 = construir_adj_set(arestas_g2)

    # condições necessárias: mesmo nº de vértices/arestas e mesma sequência de graus
    if len(v1) != len(v2) or len(arestas_g1) != len(arestas_g2):
        return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)
    if grau_sequencia(adj1, v1) != grau_sequencia(adj2, v2):
        return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)

    # busca por bijeção via backtracking com poda por adjacência (viável p/ ~10-20 vértices)
    v1_list, v2_list = list(v1), list(v2)
    mapeamento, usados = {}, set()

    def backtrack(i):
        if i == len(v1_list):
            return True
        u = v1_list[i]
        for cand in v2_list:
            if cand in usados:
                continue
            valido = True
            for w in v1_list[:i]:
                if (w in adj1[u]) != (mapeamento[w] in adj2[cand]):
                    valido = False
                    break
            if valido:
                mapeamento[u] = cand; usados.add(cand)
                if backtrack(i + 1):
                    return True
                usados.remove(cand); del mapeamento[u]
        return False

    if backtrack(0):
        return True, dict(mapeamento), grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)
    return False, None, grau_sequencia(adj1, v1), grau_sequencia(adj2, v2)

# 3. EXECUÇÃO
resultado, mapa, seq1, seq2 = sao_isomorfos(ARESTAS_G1, ARESTAS_G2)

print("=== VERIFICAÇÃO DE ISOMORFISMO ENTRE G1 E G2 ===")
print(f"Sequência de graus G1: {seq1}")
print(f"Sequência de graus G2: {seq2}")
if resultado:
    print("\nResultado: G1 e G2 SÃO ISOMORFOS.")
    print("Mapeamento (bijeção) encontrado:")
    for k in sorted(mapa.keys()):
        print(f"  {k}  ->  {mapa[k]}")
else:
    print("\nResultado: G1 e G2 NÃO são isomorfos.")

=== VERIFICAÇÃO DE ISOMORFISMO ENTRE G1 E G2 ===
Sequência de graus G1: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Sequência de graus G2: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Resultado: G1 e G2 SÃO ISOMORFOS.
Mapeamento (bijeção) encontrado:
  0  ->  10
  1  ->  11
  2  ->  12
  3  ->  13
  4  ->  14
  5  ->  15
  6  ->  16
  7  ->  17
  8  ->  18
  9  ->  19


In [6]:
#==========10==========
import math

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere os vértices e arestas aqui)
# =====================================================================

# Grafos com 10 vértices (identificadores de 0 a 9)
# Insira as arestas como tuplas (origem, destino)

ARESTAS_G1 = [
    (0, 1), (0, 2), (1, 3), (2, 3), (3, 4),
    (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (0, 9)
]

ARESTAS_G2 = [
    (0, 1), (0, 2), (1, 4), (2, 3), (3, 5),
    (5, 6), (6, 7), (7, 9), (8, 9), (0, 9), (1, 9)
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA (Não é necessário alterar)
# =====================================================================

def padronizar_arestas(lista_arestas):
    """
    Garante que arestas não direcionadas sejam comparadas corretamente.
    Ex: trata (1, 2) e (2, 1) como a mesma aresta estrutural.
    """
    return set(tuple(sorted(aresta)) for aresta in lista_arestas)

def calcular_metricas(e1, e2):
    conjunto1 = padronizar_arestas(e1)
    conjunto2 = padronizar_arestas(e2)

    intersecao = len(conjunto1.intersection(conjunto2))
    uniao = len(conjunto1.union(conjunto2))
    tam1 = len(conjunto1)
    tam2 = len(conjunto2)

    # 1. Índice de Jaccard
    jaccard = intersecao / uniao if uniao != 0 else 0.0

    # 2. Similaridade do Cosseno
    cosseno = intersecao / math.sqrt(tam1 * tam2) if (tam1 * tam2) != 0 else 0.0

    # 3. Coeficiente de Sobreposição
    min_len = min(tam1, tam2)
    overlap = intersecao / min_len if min_len != 0 else 0.0

    return jaccard, cosseno, overlap

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

jaccard_val, cosseno_val, overlap_val = calcular_metricas(ARESTAS_G1, ARESTAS_G2)

print("=== RESULTADOS DA SIMILARIDADE ENTRE G1 E G2 ===")
print(f"1. Coeficiente de Jaccard:      {jaccard_val:.4f}")
print(f"2. Similaridade do Cosseno:     {cosseno_val:.4f}")
print(f"3. Coeficiente de Sobreposição: {overlap_val:.4f}")

=== RESULTADOS DA SIMILARIDADE ENTRE G1 E G2 ===
1. Coeficiente de Jaccard:      0.4667
2. Similaridade do Cosseno:     0.6364
3. Coeficiente de Sobreposição: 0.6364


In [7]:
#========11========

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere seu grafo G(V, E) aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5, 6, 7]

# Lista de Arestas (Grafos não-direcionados)
ARESTAS = [
    (0, 1), (1, 2), (2, 3), (3, 0),  # Ciclo de tamanho 4 (0-1-2-3-0)
    (2, 4), (4, 5), (5, 6), (6, 2),  # Ciclo de tamanho 4 (2-4-5-6-2)
    (0, 5), (3, 6), (6, 7), (7, 3)   # Adiciona conexões que geram ciclos maiores e menores
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA
# =====================================================================

def construir_lista_adjacencia(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        if v not in adj[u]: adj[u].append(v)
        if u not in adj[v]: adj[v].append(u)
    return adj

def calcular_cintura(adj):
    """Encontra o menor ciclo (Cintura) usando Busca em Largura (BFS)"""
    menor_ciclo = float('inf')

    for inicio in adj:
        distancias = {inicio: 0}
        # Fila armazena: (vertice_atual, vertice_pai)
        fila = deque([(inicio, -1)])

        while fila:
            atual, pai = fila.popleft()

            for vizinho in adj[atual]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[atual] + 1
                    fila.append((vizinho, atual))
                elif vizinho != pai:
                    # Encontrou uma aresta cruzada que fecha um ciclo
                    tamanho_ciclo = distancias[atual] + distancias[vizinho] + 1
                    menor_ciclo = min(menor_ciclo, tamanho_ciclo)

    return menor_ciclo if menor_ciclo != float('inf') else None

def calcular_circunferencia(adj):
    """Encontra o maior ciclo (Circunferência) usando DFS com Backtracking"""
    maior_ciclo = 0

    def dfs_backtracking(atual, inicio, visitados, comprimento, pai):
        nonlocal maior_ciclo
        visitados.add(atual)

        for vizinho in adj[atual]:
            # Se o vizinho é o início e não é de onde viemos, fechamos um ciclo
            if vizinho == inicio and vizinho != pai and comprimento >= 3:
                maior_ciclo = max(maior_ciclo, comprimento)
            # Se não visitamos o vizinho nesta rota, continuamos aprofundando
            elif vizinho not in visitados:
                dfs_backtracking(vizinho, inicio, visitados, comprimento + 1, atual)

        visitados.remove(atual) # Backtracking para permitir explorar outras rotas

    for vertice in adj:
        dfs_backtracking(vertice, vertice, set(), 1, -1)

    return maior_ciclo if maior_ciclo >= 3 else None

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

adjacencia = construir_lista_adjacencia(VERTICES, ARESTAS)
cintura = calcular_cintura(adjacencia)
circunferencia = calcular_circunferencia(adjacencia)

print("=== MÉTRICAS DO GRAFO ===")
if cintura is None or circunferencia is None:
    print("O grafo é Acíclico (não possui ciclos).")
    print("Cintura: Infinito")
    print("Circunferência: 0")
else:
    print(f"Cintura (Menor Ciclo): {cintura}")
    print(f"Circunferência (Maior Ciclo): {circunferencia}")

=== MÉTRICAS DO GRAFO ===
Cintura (Menor Ciclo): 3
Circunferência (Maior Ciclo): 8


In [8]:
#==========12===========

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere o grafo e o vértice alvo aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5]

# Lista de Arestas
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (4, 5)
]

# Vértice a ser analisado
VERTICE_ALVO = 0


# =====================================================================
# 2. LÓGICA MATEMÁTICA
# =====================================================================

def calcular_excentricidade(vertices, arestas, target):
    # Construção da Lista de Adjacência
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)  # Remova esta linha se o grafo for direcionado

    if target not in adj:
        return None, {}

    # BFS a partir do vértice alvo para encontrar as menores distâncias
    distancias = {v: float('inf') for v in vertices}
    distancias[target] = 0
    fila = deque([target])

    while fila:
        atual = fila.popleft()
        for vizinho in adj[atual]:
            if distancias[vizinho] == float('inf'):
                distancias[vizinho] = distancias[atual] + 1
                fila.append(vizinho)

    # A excentricidade é a maior distância encontrada
    excentricidade = max(distancias.values())
    return excentricidade, distancias

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

exc, dists = calcular_excentricidade(VERTICES, ARESTAS, VERTICE_ALVO)

print(f"=== RESULTADO PARA O VÉRTICE {VERTICE_ALVO} ===")
if exc == float('inf'):
    print(f"Excentricidade e({VERTICE_ALVO}): Infinito (O grafo é desconexo ou o vértice não alcança todos os nós).")
else:
    print(f"Excentricidade e({VERTICE_ALVO}): {exc}")

print("\nDistâncias mínimas calculadas:")
for v, d in dists.items():
    print(f"  d({VERTICE_ALVO}, {v}) = {d}")

=== RESULTADO PARA O VÉRTICE 0 ===
Excentricidade e(0): 4

Distâncias mínimas calculadas:
  d(0, 0) = 0
  d(0, 1) = 1
  d(0, 2) = 1
  d(0, 3) = 2
  d(0, 4) = 3
  d(0, 5) = 4


In [9]:
#==============13============

from collections import deque

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere o grafo a ser analisado aqui)
# =====================================================================

# Lista de Vértices
VERTICES = [0, 1, 2, 3, 4, 5]

# Lista de Arestas (Grafo não-direcionado)
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (4, 5)
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA (Não é necessário alterar)
# =====================================================================

def bfs_excentricidade(start, adj, vertices):
    distancias = {v: float('inf') for v in vertices}
    distancias[start] = 0
    fila = deque([start])

    while fila:
        atual = fila.popleft()
        for vizinho in adj[atual]:
            if distancias[vizinho] == float('inf'):
                distancias[vizinho] = distancias[atual] + 1
                fila.append(vizinho)

    return max(distancias.values())

def calcular_propriedades_grafo(vertices, arestas):
    adj = {v: [] for v in vertices}
    for u, v in arestas:
        adj[u].append(v)
        adj[v].append(u)  # Remova esta linha se o grafo for direcionado

    excentricidades = {}
    for v in vertices:
        excentricidades[v] = bfs_excentricidade(v, adj, vertices)

    # Se houver vértices inalcançáveis (grafo desconexo)
    if any(e == float('inf') for e in excentricidades.values()):
        return float('inf'), float('inf'), [], excentricidades

    raio = min(excentricidades.values())
    diametro = max(excentricidades.values())
    centro = [v for v, e in excentricidades.items() if e == raio]

    return raio, diametro, centro, excentricidades

# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

raio, diametro, centro, excs = calcular_propriedades_grafo(VERTICES, ARESTAS)

print("=== PROPRIEDADES DO GRAFO G(V, E) ===")
if raio == float('inf'):
    print("O grafo é desconexo (possuindo distâncias infinitas entre vértices).")
else:
    print(f"Raio r(G):     {raio}")
    print(f"Diâmetro d(G): {diametro}")
    print(f"Centro C(G):   {centro}")

print("\nExcentricidades individuais:")
for v, e in excs.items():
    print(f"  e({v}) = {e}")

=== PROPRIEDADES DO GRAFO G(V, E) ===
Raio r(G):     2
Diâmetro d(G): 4
Centro C(G):   [3]

Excentricidades individuais:
  e(0) = 4
  e(1) = 3
  e(2) = 3
  e(3) = 2
  e(4) = 3
  e(5) = 4


In [10]:
#========14=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = [0,1,2,3,4,5,6,7]
ARESTAS = [(0,1),(1,2),(2,0),(1,3),(3,4),(4,5),(5,3),(4,6),(6,7)]

# 2. LÓGICA (Algoritmo de Tarjan: DFS com tempo de descoberta e low-link)
def encontrar_cortes(vertices, arestas):
    adj = construir_adjacencia(vertices, arestas)  # reaproveitando a função da Q8
    descoberta, low = {}, {}
    pai = {v: None for v in vertices}
    visitado = set()
    tempo = [0]
    vertices_corte = set()
    arestas_corte = []

    def dfs(u):
        visitado.add(u)
        descoberta[u] = low[u] = tempo[0]; tempo[0] += 1
        filhos = 0
        for v in adj[u]:
            if v not in visitado:
                filhos += 1
                pai[v] = u
                dfs(v)
                low[u] = min(low[u], low[v])
                if low[v] > descoberta[u]:          # ponte
                    arestas_corte.append((u, v))
                if pai[u] is None and filhos > 1:    # raiz com 2+ filhos
                    vertices_corte.add(u)
                if pai[u] is not None and low[v] >= descoberta[u]:  # articulação
                    vertices_corte.add(u)
            elif v != pai[u]:
                low[u] = min(low[u], descoberta[v])

    for v in vertices:
        if v not in visitado:
            dfs(v)
    return sorted(vertices_corte), arestas_corte

# 3. EXECUÇÃO
v_corte, a_corte = encontrar_cortes(VERTICES, ARESTAS)

print("=== CORTE EM VÉRTICES E ARESTAS DE G(V,E) ===")
print(f"Vértices de corte (pontos de articulação): {v_corte}")
print(f"Arestas de corte (pontes): {a_corte}")

=== CORTE EM VÉRTICES E ARESTAS DE G(V,E) ===
Vértices de corte (pontos de articulação): [1, 3, 4, 6]
Arestas de corte (pontes): [(6, 7), (4, 6), (1, 3)]


In [11]:
#========15=========
from collections import defaultdict, deque

# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(10))
ARESTAS = [(0,1),(0,2),(1,3),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(0,9),(2,5),(4,8)]

# 2. LÓGICA
def gerar_arvore_geradora(vertices, arestas):
    adj = construir_adjacencia(vertices, arestas)  # reaproveitando função da Q8
    visitado = set()
    arestas_arvore = []
    def dfs(u):
        visitado.add(u)
        for v in adj[u]:
            if v not in visitado:
                arestas_arvore.append((u, v))
                dfs(v)
    for v in vertices:
        if v not in visitado:
            dfs(v)
    return arestas_arvore

def cortes_fundamentais(vertices, arestas, arvore):
    """Para cada aresta da árvore geradora, remove ela e encontra as duas
    partições de vértices resultantes. O corte fundamental é o conjunto de
    todas as arestas de G que cruzam entre essas duas partições."""
    adj_arvore_base = defaultdict(list)
    for u, v in arvore:
        adj_arvore_base[u].append(v)
        adj_arvore_base[v].append(u)

    resultado = {}
    for (u, v) in arvore:
        adj_temp = defaultdict(list, {k: list(vs) for k, vs in adj_arvore_base.items()})
        adj_temp[u].remove(v)
        adj_temp[v].remove(u)

        # BFS a partir de u na árvore SEM a aresta (u,v) -> acha o lado A
        visitado = {u}
        fila = deque([u])
        while fila:
            atual = fila.popleft()
            for prox in adj_temp[atual]:
                if prox not in visitado:
                    visitado.add(prox)
                    fila.append(prox)
        lado_A = visitado
        lado_B = set(vertices) - lado_A

        corte = [(a, b) for (a, b) in arestas
                 if (a in lado_A and b in lado_B) or (a in lado_B and b in lado_A)]
        resultado[(u, v)] = corte
    return resultado

# 3. EXECUÇÃO
arvore = gerar_arvore_geradora(VERTICES, ARESTAS)
cortes = cortes_fundamentais(VERTICES, ARESTAS, arvore)

print("=== CORTE FUNDAMENTAL DE G(V,E) ===")
print(f"Árvore geradora usada: {arvore}\n")
for aresta_arvore, corte in cortes.items():
    print(f"  Corte fundamental de {aresta_arvore}: {corte}")

=== CORTE FUNDAMENTAL DE G(V,E) ===
Árvore geradora usada: [(0, 1), (1, 3), (3, 2), (2, 5), (5, 4), (4, 8), (8, 7), (7, 6), (8, 9)]

  Corte fundamental de (0, 1): [(0, 1), (0, 2), (0, 9)]
  Corte fundamental de (1, 3): [(0, 2), (1, 3), (0, 9)]
  Corte fundamental de (3, 2): [(0, 2), (2, 3), (3, 4), (0, 9)]
  Corte fundamental de (2, 5): [(3, 4), (0, 9), (2, 5)]
  Corte fundamental de (5, 4): [(3, 4), (4, 5), (5, 6), (0, 9)]
  Corte fundamental de (4, 8): [(5, 6), (0, 9), (4, 8)]
  Corte fundamental de (8, 7): [(5, 6), (7, 8)]
  Corte fundamental de (7, 6): [(5, 6), (6, 7)]
  Corte fundamental de (8, 9): [(8, 9), (0, 9)]


In [12]:
#========16=========
# pip install networkx   (se ainda não tiver instalado)
import networkx as nx

# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(6))
ARESTAS = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3),(4,5)]  # exemplo planar (K4 + aresta solta)

# 2. LÓGICA
def verificar_planaridade(vertices, arestas):
    V, E = len(vertices), len(arestas)
    condicao_euler = (E <= 3*V - 6) if V >= 3 else True  # condição necessária p/ grafo simples

    G = nx.Graph()
    G.add_nodes_from(vertices)
    G.add_edges_from(arestas)
    eh_planar, _ = nx.check_planarity(G)  # teste definitivo (algoritmo Left-Right)
    return eh_planar, condicao_euler, V, E

# 3. EXECUÇÃO
eh_planar, cond_euler, V, E = verificar_planaridade(VERTICES, ARESTAS)

print("=== VERIFICAÇÃO DE PLANARIDADE DE G(V,E) ===")
print(f"Vértices: {V} | Arestas: {E}")
print(f"Condição necessária de Euler (E <= 3V-6): {'OK' if cond_euler else 'VIOLADA'}")
print(f"Resultado do teste de planaridade: {'G É PLANAR' if eh_planar else 'G NÃO É PLANAR'}")

=== VERIFICAÇÃO DE PLANARIDADE DE G(V,E) ===
Vértices: 6 | Arestas: 7
Condição necessária de Euler (E <= 3V-6): OK
Resultado do teste de planaridade: G É PLANAR


In [13]:
#========17=========
# 1. ÁREA DE INPUT EDITÁVEL
VERTICES = list(range(15))
ARESTAS = [
    (0,1),(0,2),(1,3),(1,4),(2,5),(2,6),
    (3,7),(4,7),(4,8),(5,8),(5,9),(6,9),
    (7,10),(8,10),(8,11),(9,11),(10,12),(11,13),
    (12,14),(13,14)
]

# 2. LÓGICA
def contrair_aresta(vertices, arestas, aresta_escolhida):
    """Funde v em u: remove a própria aresta contraída (viraria laço),
    redireciona as demais arestas de v para u e elimina arestas duplicadas."""
    u, v = aresta_escolhida
    novos_vertices = [x for x in vertices if x != v]
    novas_arestas = []
    vistas = set()
    for (a, b) in arestas:
        if (a, b) == (u, v) or (a, b) == (v, u):
            continue
        aa = u if a == v else a
        bb = u if b == v else b
        if aa == bb:
            continue  # remove laço resultante
        par = tuple(sorted((aa, bb)))
        if par not in vistas:
            vistas.add(par)
            novas_arestas.append((aa, bb))
    return novos_vertices, novas_arestas

def contracao_maxima(vertices, arestas, verboso=True):
    v_atual, a_atual = list(vertices), list(arestas)
    passo = 1
    while a_atual:
        aresta_escolhida = a_atual[0]
        v_atual, a_atual = contrair_aresta(v_atual, a_atual, aresta_escolhida)
        if verboso:
            print(f"  Passo {passo}: contraiu {aresta_escolhida} -> {len(v_atual)} vértices, {len(a_atual)} arestas restantes")
        passo += 1
    return v_atual, a_atual

# 3. EXECUÇÃO
print("=== CONTRAÇÃO MÁXIMA DE G(V,E) ===")
v_final, a_final = contracao_maxima(VERTICES, ARESTAS)
print(f"\nResultado final: {len(v_final)} vértice(s) restante(s), vértice(s): {v_final}, arestas: {a_final}")

=== CONTRAÇÃO MÁXIMA DE G(V,E) ===
  Passo 1: contraiu (0, 1) -> 14 vértices, 19 arestas restantes
  Passo 2: contraiu (0, 2) -> 13 vértices, 18 arestas restantes
  Passo 3: contraiu (0, 3) -> 12 vértices, 17 arestas restantes
  Passo 4: contraiu (0, 4) -> 11 vértices, 15 arestas restantes
  Passo 5: contraiu (0, 5) -> 10 vértices, 13 arestas restantes
  Passo 6: contraiu (0, 6) -> 9 vértices, 11 arestas restantes
  Passo 7: contraiu (0, 7) -> 8 vértices, 10 arestas restantes
  Passo 8: contraiu (0, 8) -> 7 vértices, 8 arestas restantes
  Passo 9: contraiu (0, 9) -> 6 vértices, 6 arestas restantes
  Passo 10: contraiu (0, 10) -> 5 vértices, 5 arestas restantes
  Passo 11: contraiu (0, 11) -> 4 vértices, 4 arestas restantes
  Passo 12: contraiu (0, 12) -> 3 vértices, 3 arestas restantes
  Passo 13: contraiu (0, 13) -> 2 vértices, 1 arestas restantes
  Passo 14: contraiu (0, 14) -> 1 vértices, 0 arestas restantes

Resultado final: 1 vértice(s) restante(s), vértice(s): [0], arestas: []


In [14]:
#========18=========
from collections import defaultdict

# 1. ÁREA DE INPUT EDITÁVEL — dois ciclos C10 (grafos simétricos) com rótulos diferentes
ARESTAS_G1 = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),(8,9),(9,0)]
ARESTAS_G2 = [(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),(16,17),(17,18),(18,19),(19,10)]

# 2. LÓGICA (reaproveitando sao_isomorfos da Q9 e contracao_maxima da Q17)
iso, mapa, seq1, seq2 = sao_isomorfos(ARESTAS_G1, ARESTAS_G2)   # <-- corrigido: 4 valores

vertices_g1 = list(set(x for par in ARESTAS_G1 for x in par))
vertices_g2 = list(set(x for par in ARESTAS_G2 for x in par))

v1_final, a1_final = contracao_maxima(vertices_g1, ARESTAS_G1, verboso=False)
v2_final, a2_final = contracao_maxima(vertices_g2, ARESTAS_G2, verboso=False)

# 3. EXECUÇÃO
print("=== G1 E G2: SIMÉTRICOS, ISOMORFOS E CONTRAÇÃO MÁXIMA ===")
print(f"G1 e G2 são isomorfos? {iso}")
print(f"Sequência de graus G1: {seq1} | G2: {seq2}")
print(f"Mapeamento: {mapa}\n")
print(f"Contração máxima de G1: {len(v1_final)} vértice(s) -> {v1_final}, arestas: {a1_final}")
print(f"Contração máxima de G2: {len(v2_final)} vértice(s) -> {v2_final}, arestas: {a2_final}")
equivalentes = len(v1_final) == len(v2_final) and len(a1_final) == len(a2_final) == 0
print(f"\nResultados estruturalmente equivalentes: {equivalentes}")

=== G1 E G2: SIMÉTRICOS, ISOMORFOS E CONTRAÇÃO MÁXIMA ===
G1 e G2 são isomorfos? True
Sequência de graus G1: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2] | G2: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Mapeamento: {0: 10, 1: 11, 2: 12, 3: 13, 4: 14, 5: 15, 6: 16, 7: 17, 8: 18, 9: 19}

Contração máxima de G1: 1 vértice(s) -> [0], arestas: []
Contração máxima de G2: 1 vértice(s) -> [10], arestas: []

Resultados estruturalmente equivalentes: True


In [15]:
#==============19=============

from collections import defaultdict

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere os vértices e arestas do DAG aqui)
# =====================================================================

NUM_VERTICES = 15

# Lista de Arestas Direcionadas (u -> v)
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6),
    (3, 7), (4, 7), (4, 8), (5, 8), (5, 9), (6, 9),
    (7, 10), (8, 10), (8, 11), (9, 11), (10, 12), (11, 13),
    (12, 14), (13, 14)
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA (DFS e Ordenação Topológica)
# =====================================================================

class GrafoDAG:
    def __init__(self, num_vertices):
        self.V = num_vertices
        self.adj = defaultdict(list)

    def adicionar_aresta(self, u, v):
        self.adj[u].append(v)

    def _dfs_topologica(self, v, visitado, pilha):
        visitado[v] = True

        # Travessia DFS nos vértices adjacentes
        for vizinho in self.adj[v]:
            if not visitado[vizinho]:
                self._dfs_topologica(vizinho, visitado, pilha)

        # Armazena o vértice na pós-ordem do término do processamento
        pilha.append(v)

    def obter_rotulacao_topologica(self):
        visitado = [False] * self.V
        pilha = []

        # Aplica DFS em todos os componentes do grafo
        for i in range(self.V):
            if not visitado[i]:
                self._dfs_topologica(i, visitado, pilha)

        # A ordem topológica é o inverso da pós-ordem de término da DFS
        sequencia_topologica = pilha[::-1]

        # Atribuição dos rótulos numéricos (0 a V-1) segundo a sequência válida
        rotulacao = {vertice: posicao for posicao, vertice in enumerate(sequencia_topologica)}

        return sequencia_topologica, rotulacao


# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

grafo = GrafoDAG(NUM_VERTICES)
for u, v in ARESTAS:
    grafo.adicionar_aresta(u, v)

sequencia, rotulos = grafo.obter_rotulacao_topologica()

print("=== RESULTADO DA ROTULAÇÃO TOPOLÓGICA (DFS) ===")
print(f"Sequência Válida de Travessia: {sequencia}\n")
print("Rótulos Atribuídos aos Vértices:")
for v in sorted(rotulos.keys()):
    print(f"  Vértice {v:2d} -> Rótulo Topológico: {rotulos[v]:2d}")

=== RESULTADO DA ROTULAÇÃO TOPOLÓGICA (DFS) ===
Sequência Válida de Travessia: [0, 2, 6, 5, 9, 1, 4, 8, 11, 13, 3, 7, 10, 12, 14]

Rótulos Atribuídos aos Vértices:
  Vértice  0 -> Rótulo Topológico:  0
  Vértice  1 -> Rótulo Topológico:  5
  Vértice  2 -> Rótulo Topológico:  1
  Vértice  3 -> Rótulo Topológico: 10
  Vértice  4 -> Rótulo Topológico:  6
  Vértice  5 -> Rótulo Topológico:  3
  Vértice  6 -> Rótulo Topológico:  2
  Vértice  7 -> Rótulo Topológico: 11
  Vértice  8 -> Rótulo Topológico:  7
  Vértice  9 -> Rótulo Topológico:  4
  Vértice 10 -> Rótulo Topológico: 12
  Vértice 11 -> Rótulo Topológico:  8
  Vértice 12 -> Rótulo Topológico: 13
  Vértice 13 -> Rótulo Topológico:  9
  Vértice 14 -> Rótulo Topológico: 14


In [16]:
#===========20==========

from collections import deque, defaultdict

# =====================================================================
# 1. ÁREA DE INPUT EDITÁVEL (Altere os vértices e arestas do DAG aqui)
# =====================================================================

NUM_VERTICES = 15

# Lista de Arestas Direcionadas (u -> v)
ARESTAS = [
    (0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6),
    (3, 7), (4, 7), (4, 8), (5, 8), (5, 9), (6, 9),
    (7, 10), (8, 10), (8, 11), (9, 11), (10, 12), (11, 13),
    (12, 14), (13, 14)
]


# =====================================================================
# 2. LÓGICA MATEMÁTICA (BFS / Algoritmo de Kahn)
# =====================================================================

class GrafoDAG_BFS:
    def __init__(self, num_vertices):
        self.V = num_vertices
        self.adj = defaultdict(list)
        self.in_degree = [0] * num_vertices

    def adicionar_aresta(self, u, v):
        self.adj[u].append(v)
        self.in_degree[v] += 1  # Incrementa o grau de entrada do destino

    def obter_rotulacao_topologica(self):
        # Fila para BFS: Inicializada com vértices de grau de entrada 0 (sem dependências)
        fila = deque([v for v in range(self.V) if self.in_degree[v] == 0])

        sequencia_topologica = []
        grau_entrada = list(self.in_degree)

        while fila:
            u = fila.popleft()
            sequencia_topologica.append(u)

            # Para cada vizinho, simula a remoção da aresta (u -> v)
            for v in self.adj[u]:
                grau_entrada[v] -= 1
                # Se todas as dependências de v foram processadas, entra na fila
                if grau_entrada[v] == 0:
                    fila.append(v)

        # Validação: se o tamanho da sequência for menor que V, o grafo possui ciclos
        if len(sequencia_topologica) != self.V:
            return None, {}

        # Mapeamento dos rótulos numéricos (0 a V-1)
        rotulacao = {vertice: posicao for posicao, vertice in enumerate(sequencia_topologica)}

        return sequencia_topologica, rotulacao


# =====================================================================
# 3. EXECUÇÃO
# =====================================================================

grafo = GrafoDAG_BFS(NUM_VERTICES)
for u, v in ARESTAS:
    grafo.adicionar_aresta(u, v)

sequencia, rotulos = grafo.obter_rotulacao_topologica()

print("=== RESULTADO DA ROTULAÇÃO TOPOLÓGICA (BFS - Kahn) ===")
if sequencia is None:
    print("Erro: O grafo contém um ciclo (não é um DAG).")
else:
    print(f"Sequência Válida de Travessia: {sequencia}\n")
    print("Rótulos Atribuídos aos Vértices:")
    for v in sorted(rotulos.keys()):
        print(f"  Vértice {v:2d} -> Rótulo Topológico: {rotulos[v]:2d}")

=== RESULTADO DA ROTULAÇÃO TOPOLÓGICA (BFS - Kahn) ===
Sequência Válida de Travessia: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Rótulos Atribuídos aos Vértices:
  Vértice  0 -> Rótulo Topológico:  0
  Vértice  1 -> Rótulo Topológico:  1
  Vértice  2 -> Rótulo Topológico:  2
  Vértice  3 -> Rótulo Topológico:  3
  Vértice  4 -> Rótulo Topológico:  4
  Vértice  5 -> Rótulo Topológico:  5
  Vértice  6 -> Rótulo Topológico:  6
  Vértice  7 -> Rótulo Topológico:  7
  Vértice  8 -> Rótulo Topológico:  8
  Vértice  9 -> Rótulo Topológico:  9
  Vértice 10 -> Rótulo Topológico: 10
  Vértice 11 -> Rótulo Topológico: 11
  Vértice 12 -> Rótulo Topológico: 12
  Vértice 13 -> Rótulo Topológico: 13
  Vértice 14 -> Rótulo Topológico: 14
